<a href="https://colab.research.google.com/github/BasuAI/BasuAI/blob/main/EDAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [60]:
df = pd.read_csv('/content/EDAS DATA.csv', encoding='latin1')

In [61]:
df.head()

,FID,Clay,Sand,DD,Elevation,Slope,NDVI,NDWI,Rainfall,Mean temp,Curveture
0,0,402,86,0.23,5,2.94,0.58,0.37,1207.36,25.67,0.0000
1,1,389,95,0.00,7,3.67,0.39,0.24,1207.73,25.67,0.0000
2,2,397,87,0.00,9,2.63,0.51,0.31,1208.01,25.67,0.4444
3,3,387,91,0.84,4,0.00,0.55,0.31,1202.95,25.67,0.0000
4,4,393,91,0.63,8,2.32,0.57,0.33,1203.73,25.67,0.3333


In [62]:
weights_dict = {
    "Clay":       0.088193,
    "Sand":       0.090915,
    "DD":         0.250093,
    "Elevation":  0.062652,
    "Slope":      0.527658,
    "NDVI":       0.048602,
    "NDWI":       0.040722,
    "Rainfall":   0.013713,
    "Mean temp":   0.013854,
    "Curveture":  -0.136402
}

In [63]:
beneficial_criteria = ["Sand", "DD", "NDVI", "NDWI", "Rainfall"]
cost_criteria = ["Clay", "Elevation", "Slope", "Mean temp", "Curveture"]

criteria_cols = list(weights_dict.keys())

In [64]:
AS = df[criteria_cols].mean()

In [65]:
df_EDAS = df.copy()

for c in criteria_cols:
    if c in beneficial_criteria:
        # For beneficial criteria:
        df_EDAS[f"PD_{c}"] = np.maximum(0, (df[c] - AS[c]) / AS[c])
        df_EDAS[f"ND_{c}"] = np.maximum(0, (AS[c] - df[c]) / AS[c])
    else:
        # For cost criteria:
        df_EDAS[f"PD_{c}"] = np.maximum(0, (AS[c] - df[c]) / AS[c])
        df_EDAS[f"ND_{c}"] = np.maximum(0, (df[c] - AS[c]) / AS[c])


In [66]:
df_EDAS["SP"] = 0.0
df_EDAS["SN"] = 0.0

for c in criteria_cols:
    w = weights_dict[c]
    df_EDAS["SP"] += w * df_EDAS[f"PD_{c}"]
    df_EDAS["SN"] += w * df_EDAS[f"ND_{c}"]

In [67]:
SP_min, SP_max = df_EDAS["SP"].min(), df_EDAS["SP"].max()
SN_min, SN_max = df_EDAS["SN"].min(), df_EDAS["SN"].max()

df_EDAS["NSP"] = (df_EDAS["SP"] - SP_min) / (SP_max - SP_min + 1e-14)
df_EDAS["NSN"] = (SN_max - df_EDAS["SN"]) / (SN_max - SN_min + 1e-14)

In [68]:
df_EDAS["AppraisalScore"] = 0.5 * (df_EDAS["NSP"] + df_EDAS["NSN"])
df_EDAS["Rank"] = df_EDAS["AppraisalScore"].rank(ascending=False, method="dense")

In [69]:
cols_to_export = [
    "FID", "SP", "SN", "NSP", "NSN", "AppraisalScore", "Rank"
]
df_results = df_EDAS[cols_to_export].sort_values("Rank")
df_results.to_csv("EDAS_results.csv", index=False)

print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [70]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [71]:
print(df.head())

   FID  Clay  Sand    DD  Elevation  Slope  NDVI  NDWI  Rainfall  Mean temp  \
0    0   402    86  0.23          5   2.94  0.58  0.37   1207.36      25.67   
1    1   389    95  0.00          7   3.67  0.39  0.24   1207.73      25.67   
2    2   397    87  0.00          9   2.63  0.51  0.31   1208.01      25.67   
3    3   387    91  0.84          4   0.00  0.55  0.31   1202.95      25.67   
4    4   393    91  0.63          8   2.32  0.57  0.33   1203.73      25.67   

   Curveture  
0     0.0000  
1     0.0000  
2     0.4444  
3     0.0000  
4     0.3333  


In [73]:
X = df_EDAS[['SP', 'SN', 'NSP', 'NSN']]
y = df_EDAS['AppraisalScore']

In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [75]:
y_pred = rf_model.predict(X_test)


In [76]:
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)


In [77]:
print("Random Forest Regressor Performance:")
print("R^2 Score:", r2)
print("Mean Squared Error:", mse)


Random Forest Regressor Performance:
R^2 Score: 0.9506216600089429
Mean Squared Error: 0.0011992146385054034


In [78]:
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nFeature Importances:")
print(feature_importances.sort_values(ascending=False))


Feature Importances:
NSN    0.347717
SN     0.240280
SP     0.210846
NSP    0.201157
dtype: float64
